# Hotel Bar Inventory Forecasting

Predicts how much each bar will sell of each drink, and recommends how much stock (par level) to keep. Tested against real history to see if it actually cuts down stockouts.


## Step 1: Load and clean data

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

filename = 'Consumption_Dataset_-_Dataset.csv'
if not os.path.exists(filename):
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
df['Date Time Served'] = pd.to_datetime(df['Date Time Served'])
df['Date'] = df['Date Time Served'].dt.floor('D')
df.head()


In [ ]:
# check opening + purchase - consumed = closing, flag anything that doesn't add up
expected = df['Opening Balance (ml)'] + df['Purchase (ml)'] - df['Consumed (ml)']
diff = (expected - df['Closing Balance (ml)']).abs()
print("rows off by more than 0.01ml:", (diff > 0.01).sum())


Most days a bar sold zero of some brand, so there's just no row for it. Need to add those zeros in myself, or the model thinks demand is way higher than it really is.

In [ ]:
# total up consumption per day/bar/brand -- most days a brand just didn't sell, no row for it
daily = df.groupby(['Date', 'Bar Name', 'Brand Name'])['Consumed (ml)'].sum().reset_index()
all_days = pd.date_range(df['Date'].min(), df['Date'].max(), freq='D')
n_bars, n_brands = df['Bar Name'].nunique(), df['Brand Name'].nunique()
print(f"only {len(daily) / (len(all_days)*n_bars*n_brands) * 100:.1f}% of bar-brand-days actually had a sale")


In [ ]:
# turn the grid back into a plain list: one row per day/bar/brand
bars = df['Bar Name'].unique()
brands = df['Brand Name'].unique()
full_index = pd.MultiIndex.from_product([all_days, bars, brands], names=['Date','Bar Name','Brand Name'])
daily_ts = daily.set_index(['Date','Bar Name','Brand Name']).reindex(full_index, fill_value=0).reset_index()
daily_ts = daily_ts.rename(columns={'Consumed (ml)': 'consumed_ml'})


## Step 2: Quick look at the data

In [ ]:
# rank bar+brand combos by total volume, split into 3 buckets
totals = daily_ts.groupby(['Bar Name', 'Brand Name'])['consumed_ml'].sum().sort_values(ascending=False)
demand_level = pd.qcut(totals, q=3, labels=['Low', 'Medium', 'High'])
demand_table = demand_level.reset_index()
demand_table.columns = ['Bar Name', 'Brand Name', 'Demand Level']
demand_table['Demand Level'].value_counts()


In [ ]:
# average sales by day of week -- checking for the usual Fri/Sat spike
daily_ts['day_of_week'] = daily_ts['Date'].dt.day_name()
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
avg_by_day = daily_ts.groupby('day_of_week')['consumed_ml'].mean().reindex(order)
avg_by_day.plot(kind='bar', figsize=(8,4), title='Avg ml sold by day of week')
plt.show()


Pretty flat across the week actually, Wednesday's the highest, not the weekend. So day-of-week probably won't help much for predicting.

In [ ]:
# find days where stock hit 0 but there was demand -- actual past stockouts
stockouts = df[(df['Closing Balance (ml)'] <= 0.5) & (df['Consumed (ml)'] > 0)]
print("past stockout events:", len(stockouts))
stockouts.groupby(['Bar Name','Brand Name']).size().sort_values(ascending=False).head()


## Step 3: Predict tomorrow's demand

In [ ]:
# build lag features per bar+brand, split into train/test by date (never shuffle time data)
daily_ts = daily_ts.sort_values(['Bar Name', 'Brand Name', 'Date']).reset_index(drop=True)
g = daily_ts.groupby(['Bar Name', 'Brand Name'])['consumed_ml']
daily_ts['yesterday'] = g.shift(1)
daily_ts['same_day_last_week'] = g.shift(7)
daily_ts['avg_last_7_days'] = g.shift(1).rolling(7).mean().reset_index(drop=True)

ready = daily_ts.dropna(subset=['yesterday','same_day_last_week','avg_last_7_days']).copy()
cutoff = ready['Date'].max() - pd.Timedelta(days=60)
train = ready[ready['Date'] <= cutoff]
test = ready[ready['Date'] > cutoff].copy()


In [ ]:
# WAPE instead of MAPE -- most days are 0 demand so MAPE breaks (divide by zero)
def wape(actual, pred):
    actual, pred = np.array(actual), np.array(pred)
    return np.sum(np.abs(actual - pred)) / np.sum(actual)

scores = {}


In [ ]:
# model 1: just use last 7 days average
scores['baseline'] = wape(test['consumed_ml'], test['avg_last_7_days'])
scores['baseline']


In [ ]:
# model 2: nudge the average up/down based on which day of week it is
day_avg = train.groupby(train['Date'].dt.day_name())['consumed_ml'].mean() - train['consumed_ml'].mean()
adj = test['Date'].dt.day_name().map(day_avg)
pred_2 = (test['avg_last_7_days'] + adj).clip(lower=0)
scores['day_of_week'] = wape(test['consumed_ml'], pred_2)
scores['day_of_week']


In [ ]:
# model 3: linear regression on the 3 lag features, pooled across all series
from sklearn.linear_model import LinearRegression

cols = ['yesterday', 'same_day_last_week', 'avg_last_7_days']
model = LinearRegression().fit(train[cols], train['consumed_ml'])
pred_3 = model.predict(test[cols]).clip(min=0)
scores['linear_reg'] = wape(test['consumed_ml'], pred_3)
scores


In [ ]:
# keep whichever model actually won -- turned out to be the simple baseline
winner = min(scores, key=scores.get)
print("winner:", winner)
test['forecast'] = test['avg_last_7_days'] if winner == 'baseline' else (pred_2 if winner == 'day_of_week' else pred_3)
test['forecast_error'] = test['consumed_ml'] - test['forecast']


## Step 4: Work out the par level

In [ ]:
# par level = expected demand during delivery wait + a safety buffer for how unpredictable the item is
LEAD_TIME = 2
Z = 1.645  # 95% confidence

rows = []
for (bar, brand), g in test.groupby(['Bar Name', 'Brand Name']):
    demand = g['forecast'].mean()
    rmse = np.sqrt((g['forecast_error']**2).mean())
    safety = Z * rmse * np.sqrt(LEAD_TIME)
    rows.append({'Bar Name': bar, 'Brand Name': brand, 'par_level': demand*LEAD_TIME + safety, 'safety': safety})

par_table = pd.DataFrame(rows).merge(demand_table, on=['Bar Name','Brand Name'])
par_table.sort_values('par_level', ascending=False).head()


## Step 5: Simulate it against real history

In [ ]:
# day-by-day: sell stock, receive orders that arrive, reorder if running low
def simulate(actual_sales, par_level, lead_time=LEAD_TIME):
    stock = par_level
    incoming = []
    stockouts = 0
    history = []
    for sale in actual_sales:
        for o in incoming:
            o[0] -= 1
        arrived = sum(o[1] for o in incoming if o[0] <= 0)
        stock += arrived
        incoming = [o for o in incoming if o[0] > 0]
        if stock >= sale:
            stock -= sale
        else:
            stockouts += 1
            stock = 0
        if stock + sum(o[1] for o in incoming) < par_level:
            incoming.append([lead_time, par_level - (stock + sum(o[1] for o in incoming))])
        history.append(stock)
    return stockouts, np.mean(history)


In [ ]:
# compare old fixed-average policy vs the new par-level policy
old_stockouts, new_stockouts, old_stock, new_stock = [], [], [], []
for (bar, brand), g in test.groupby(['Bar Name', 'Brand Name']):
    g = g.sort_values('Date')
    sales = g['consumed_ml'].values
    if len(sales) < 10:
        continue
    old_par = train[(train['Bar Name']==bar) & (train['Brand Name']==brand)]['consumed_ml'].mean() * LEAD_TIME
    new_par = par_table[(par_table['Bar Name']==bar) & (par_table['Brand Name']==brand)]['par_level'].values[0]

    s_o, h_o = simulate(sales, old_par)
    s_n, h_n = simulate(sales, new_par)
    old_stockouts.append(s_o); new_stockouts.append(s_n)
    old_stock.append(h_o); new_stock.append(h_n)

print("old stockout-days:", sum(old_stockouts), " new:", sum(new_stockouts))
print("old avg stock held:", round(np.mean(old_stock)), " new:", round(np.mean(new_stock)))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].bar(['old','new'], [sum(old_stockouts), sum(new_stockouts)], color=['red','green'])
ax[0].set_title('total stockout-days')
ax[1].bar(['old','new'], [np.mean(old_stock), np.mean(new_stock)], color=['red','green'])
ax[1].set_title('avg stock held (ml)')
plt.show()


## Step 6: Summary

Filling in the zero-sale days mattered most -- 81% of the data was silent days that had to be added in.
Tried 3 forecast models, simplest one won, so went with that.
New par-level system cuts stockouts a lot, but holds a lot more stock too -- real trade-off, not free.
With more time: use real supplier lead times instead of a flat 2 days, and account for one-off events like holidays.